## Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.2.1</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1</strong></li>
</ul>
</div>

# Lab 2 – Fine-Tune an LLM with Serverless Customization

In **Lab 1** we curated a medical reasoning dataset and registered it in the SageMaker AI Registry. In this lab we run the actual **Supervised Fine-Tuning (SFT)** job that adapts a general-purpose base model to that domain.

**What "serverless customization" means here:** instead of provisioning, configuring, and tearing down a training cluster yourself, you submit a high-level `SFTTrainer` job and SageMaker AI provisions the compute, runs the training recipe, and releases the resources when it finishes — you only pay for what the job uses. See [Customizing models with Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/customize-model.html) for the full picture of the serverless customization capability.

**What this notebook does, step by step:**

1. Pick a base model from [SageMaker JumpStart](https://docs.aws.amazon.com/sagemaker/latest/dg/jumpstart-foundation-models.html).
2. Create a **Model Package Group** so every model we train is versioned in one place in the [Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html).
3. Configure an `SFTTrainer` job that uses **LoRA** (a parameter-efficient fine-tuning technique) and references the registered datasets from Lab 1.
4. Inspect and override the training hyperparameters.
5. Submit the job and track it to completion.

> The fine-tuned model produced here is what **Lab 3** deploys to a real-time endpoint.

## Step 1 – Choose a base model

The base model used across this lab is configured in [`config.py`](config.py) and defaults to a Qwen 3 reasoning model. We fine-tune a **pre-trained foundation model** rather than training from scratch: the base model already understands language and reasoning, and SFT simply nudges it toward our medical-reasoning style and format.

Want to try a different model? The cell below lists every [SageMaker JumpStart](https://docs.aws.amazon.com/sagemaker/latest/dg/jumpstart-foundation-models.html) model that supports customization. To switch models, update `BASE_MODEL_ID` in `config.py` — every notebook in this lab reads from there, so the change propagates automatically.

> **Note:** Not every JumpStart model supports every customization technique (SFT, DPO, RLVR, …). If you hit a *"No recipes found"* error, that model does not support the technique used in this lab. See [Customizing models with Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/customize-model.html) for which techniques apply to which models.

In [ ]:
# --- Lab dependencies (managed via uv) ---------------------------------------
# Installs THIS lab's complete, self-contained kernel dependencies from the
# lab requirements.txt using uv. Idempotent and fast when already satisfied.
# This is the only dependency step the lab needs - Setup.ipynb is not required.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r requirements.txt

In [ ]:
import boto3
from config import BASE_MODEL_ID

# Retrieve all JumpStart models that support customization (fine-tuning)
sm = boto3.client("sagemaker")
models = []
kwargs = {"HubName": "SageMakerPublicHub", "HubContentType": "Model", "MaxResults": 100}
while True:
    response = sm.list_hub_contents(**kwargs)
    for item in response["HubContentSummaries"]:
        keywords = item.get("HubContentSearchKeywords", [])
        if "@capability:customization" in keywords:
            models.append(item["HubContentName"])
    if "NextToken" in response:
        kwargs["NextToken"] = response["NextToken"]
    else:
        break

models.sort()
print(f"Current model: {BASE_MODEL_ID}\n")
print(f"Available models ({len(models)}):")
print("\n".join(models))

***

In [ ]:
%load_ext autoreload
%autoreload 2

### Prerequisites

### Step 2 – Set up the SageMaker session

The next cell establishes the SageMaker [`Session`](https://sagemaker.readthedocs.io/en/stable/api/utility/session.html), resolves the **execution role** (the IAM identity the training job assumes), and selects the **default S3 bucket** used to stage inputs and store outputs. The role must have permission to run training jobs and read/write that bucket.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
import os
from sagemaker.ai_registry.dataset import DataSet
from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

training_dataset = DataSet.get(name="medical-o1-reasoning-sft-train")
val_dataset = DataSet.get(name="medical-o1-reasoning-sft-val")

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}"

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = (
    f"https://mlflow.sagemaker.{sess.boto_region_name}.app.aws"
)

***

### Step 3 – Create a Model Package Group

A **Model Package Group** is a container in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html) that holds successive **versions** of a model. Registering each fine-tuning run as a new model package version gives you lineage, comparison, and a clean hand-off to deployment (Lab 3 pulls the model from this group). The cell is idempotent — it reuses the group if it already exists.

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name = f"{base_model_id}-sft-mpg"

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless SFT customization",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

## Configure MLflow

In [ ]:
import time
sagemaker_session = Session()

region = sagemaker_session.boto_session.region_name
sm_client = boto3.client("sagemaker", region_name=region)

mlflow_name = "mlflow-app"
apps = sm_client.list_mlflow_apps().get("Summaries", [])
mlflow_app = next((a for a in apps if a["Name"] == mlflow_name), None)

if mlflow_app:
    print(f"Using existing MLflow App: {mlflow_app['Name']}")
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app["Arn"])
else:
    print(f"Creating MLflow App: {mlflow_name}...")
    response = sm_client.create_mlflow_app(
        Name=mlflow_name,
        ArtifactStoreUri=f"s3://{bucket_name}",
        RoleArn=role,
        ModelRegistrationMode="AutoModelRegistrationEnabled",
    )
    while True:
        mlflow_app = sm_client.describe_mlflow_app(Arn=response["Arn"])
        if mlflow_app["Status"] in ["Created", "Updated"]:
            break
        elif mlflow_app["Status"] in ["CreateFailed", "Deleted"]:
            raise RuntimeError(f"MLflow App creation failed: {mlflow_app['Status']}")
        print(f"Status: {mlflow_app['Status']}... waiting")
        time.sleep(30)

while mlflow_app["Status"] in ["Creating", "Updating"]:
    print("MLflow App creating... waiting")
    time.sleep(30)
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app["Arn"])

# Variable name kept as `mlflow_tracking_server_arn` for compatibility with the rest
# of this lab and the steps/ modules (and the args.yaml heredoc below).
mlflow_tracking_server_arn = mlflow_app["Arn"]
print(f"MLflow App: {mlflow_app['Name']} (v{mlflow_app.get('MlflowVersion', 'N/A')})")
print(f"ARN: {mlflow_tracking_server_arn}")

In [ ]:
import uuid

suffix = str(uuid.uuid4())[:8]
mlflow_experiment_name = f"huggingface-reasoning-qwen3-8b-sft-mpg-{suffix}"

### Step 4 – Configure the serverless SFT job

We use the [`SFTTrainer`](https://docs.aws.amazon.com/sagemaker/latest/dg/customize-model.html) abstraction. Two key choices:

- **`TrainingType.LORA`** — Low-Rank Adaptation (LoRA) is a *parameter-efficient* fine-tuning method. Instead of updating all of the model's billions of weights, LoRA freezes the base model and trains a small set of low-rank adapter matrices. This is dramatically cheaper and faster, needs far less memory, and produces a small adapter that is later merged back into the base weights for deployment.
- **Datasets** — the `training_dataset` and `validation_dataset` are the registry entries created in Lab 1, so the job trains and validates on exactly the data we curated.

`accept_eula=True` acknowledges the base model's license, which is required before JumpStart will train it.

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
    mlflow_experiment_name=mlflow_experiment_name,
    mlflow_resource_arn=mlflow_tracking_server_arn
)

#### Inspect the default hyperparameters

Each base-model + technique combination ships with a tuned **recipe** of default hyperparameters. Print them first so you understand the starting point before changing anything.

In [ ]:
from rich import print as rprint
from rich.pretty import pprint

print("Default Finetuning options:")
pprint(trainer.hyperparameters.to_dict())

#### Override selected hyperparameters

Here we override a few values for this lab:

- **`learning_rate`** – step size for the weight updates; too high diverges, too low learns slowly.
- **`global_batch_size`** – number of examples processed before each update.
- **`max_epochs = 1`** – a single pass over the data keeps this workshop run short.
- **`lr_warmup_steps_ratio`** – fraction of steps spent gradually ramping the learning rate up at the start, which stabilizes early training.

Leave the rest at their recipe defaults unless you have a reason to change them.

In [ ]:
trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.global_batch_size = 64
trainer.hyperparameters.max_epochs = 1
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1

In [ ]:
print("\nModified/user defined options:")
pprint(trainer.hyperparameters.to_dict())

### Kick off the training job

The cell below submits the fine-tuning job asynchronously (`wait=False`), so it will return immediately without waiting for training to complete. Training typically takes **20 minutes** with the chosen dataset and hyperparameters, but will depend on the model and dataset size.

Once the job is submitted, you can track its progress in two ways:

1. **AWS Console** — Navigate to **SageMaker AI > Training > Training jobs** and search for the job name.
2. **SageMaker SDK** — Use the status-check cell further below to poll the job status programmatically.

In [ ]:
from rich import print as rprint
from rich.pretty import pprint

training_job = trainer.train(wait=False)

TRAINING_JOB_NAME = training_job.training_job_name

pprint(training_job)

In [ ]:
# The training job was submitted with wait=False, so .train() returned
# immediately. Poll its status until it reaches a terminal state, then fail
# loudly on any non-Completed outcome (mirrors the from-idea-to-production
# reference: a Stopped or Failed job raises, not only Failed).
import time
from sagemaker.core.resources import TrainingJob

training_job = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"Polling training job: {TRAINING_JOB_NAME}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    training_job.refresh()
    status = training_job.training_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status} | secondary={training_job.secondary_status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(
        f"Training job {TRAINING_JOB_NAME} ended with status '{status}': "
        f"{training_job.failure_reason}"
    )
print(f"Training job {TRAINING_JOB_NAME} completed successfully.")

In [ ]:
import mlflow

mlflow.set_tracking_uri(mlflow_tracking_server_arn)

experiment_id = mlflow.get_experiment_by_name(mlflow_experiment_name).experiment_id
# get the last run in MLflow
last_run_id = mlflow.search_runs(
    experiment_ids=[experiment_id], 
    max_results=1, 
    order_by=["attributes.start_time DESC"]
)['run_id'][0]

# get the presigned url to open the MLflow UI
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_tracking_server_arn,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

mlflow_run_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/runs/{last_run_id}/model-metrics?workspace=default"

In [ ]:
from IPython.display import Javascript, HTML

# first open the MLflow UI - you can close a new opened window
display(Javascript('window.open("{}");'.format(presigned_url)))

Open model metrics for a specific model

In [ ]:
display(Javascript('window.open("{}");'.format(mlflow_run_link)))